# Notebook 3: Read the test pages and make the Kaggle file

This reads all 208 test pages with your best run from Notebook 2 (survey spreads one page at a
time, as in training) and saves `submission.csv` in the format Kaggle wants: `page_id`, `text`,
one row per test page.

About 40 to 60 minutes on an `ml.g5.xlarge`. If it stops partway, run the cell again: it carries
on where it stopped. Run the cells from top to bottom with **Shift + Enter**. Kernel: **conda_pytorch_p310** (or another `conda_pytorch` kernel, not `conda_python3`).

## Step 1. Install the libraries

The same as Notebook 2, Step 1.

In [ ]:
%pip install -q -r requirements.txt

import importlib.metadata
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    raise SystemExit("This kernel has no PyTorch, so it can't run the model. Choose Kernel → Change Kernel, "
                     "pick conda_pytorch_p310 (or another conda_pytorch kernel), then run this cell again.")

# The model also needs torchvision in exactly the version and build made for this kernel's
# PyTorch (2.8.0 goes with 0.23.0, 2.9.1 with 0.24.1, ...). conda_pytorch kernels have it.
def torchvision_works():
    return subprocess.run([sys.executable, "-c", "import torchvision"], capture_output=True).returncode == 0

torch_version, _, build = importlib.metadata.version("torch").partition("+")
if not torchvision_works():
    _, minor, patch = (torch_version.split(".") + ["0"])[:3]
    wanted = f"torchvision==0.{int(minor) + 15}.{patch}"
    sources = ([["--index-url", f"https://download.pytorch.org/whl/{build}"]] if build else []) + [[]]
    for source in sources:
        print(f"Installing {wanted} for PyTorch {torch_version}{'+' + build if build else ''} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", wanted, *source])
        if torchvision_works():
            break
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchvision"])
    else:
        raise SystemExit("Couldn't install a torchvision that works with this kernel's PyTorch. Switch the "
                         "kernel to conda_pytorch_p310 (Kernel → Change Kernel) and run this cell again.")
print(f"PyTorch {importlib.metadata.version('torch')}, torchvision {importlib.metadata.version('torchvision')}: ready.")

## Step 2. Set up the folders

In [ ]:
import importlib.util
import os
import subprocess
import sys

# Which kernel is this? It should be conda_pytorch_p310 (or another conda_pytorch kernel).
print("Kernel environment:", os.path.basename(os.path.dirname(os.path.dirname(sys.executable))))
missing = [pkg for module, pkg in (("PIL", "pillow"), ("pandas", "pandas"))
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing", ", ".join(missing), "into this kernel ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Everything this project downloads or creates lives here, on the notebook
# instance's storage volume, so it survives Stop and Start.
WORK = "/home/ec2-user/SageMaker/churro-work"
DATA = os.path.join(WORK, "data")     # page images and lists of pages
RUNS = os.path.join(WORK, "runs")     # one folder per training run
SCRIPTS = os.path.abspath("scripts")  # the scripts folder next to this notebook
for folder in (WORK, DATA, RUNS):
    os.makedirs(folder, exist_ok=True)
sys.path.insert(0, SCRIPTS)
os.environ["HF_HOME"] = os.path.join(WORK, "hf_cache")   # the 7.5 GB model download goes here

# Your bucket name is saved here, outside the repository, so it's typed once and
# updating the code (git pull) never touches it.
import json
SETTINGS = os.path.join(WORK, "my_settings.json")
def saved_setting(name, default=""):
    try:
        return json.load(open(SETTINGS)).get(name, default)
    except (OSError, ValueError):
        return default
def save_setting(name, value):
    try:
        current = json.load(open(SETTINGS))
    except (OSError, ValueError):
        current = {}
    current[name] = value
    json.dump(current, open(SETTINGS, "w"), indent=2)
print("Working folder:", WORK)

## Step 3. Pick a run for each collection

Your runs, with each collection's dev CER. Each collection can be read by a different run: for
example a `"kade-final"` run for the Kade pages, a `"survey-final"` run for the survey pages, and a
`"full"` run for Dominy. By default the run with the best mean is used for everything; type run
names into `USE` to choose. Smoke runs aren't listed. The test pages are read exactly the way each
chosen run read its dev pages, with the same settings.

Runs trained with Kade documents held out (`kadeA-...`, `kadeB-...`) are for choosing settings.
For the test pages, use a `"kade-final"` run trained on every Kade page.


In [ ]:
import json
import pandas as pd

runs = []
for name in os.listdir(RUNS):
    path = os.path.join(RUNS, name, "summary.json")
    if os.path.exists(path) and name != "smoke":
        s = json.load(open(path))
        runs.append({"run": name, **s["dev_cer"], "dev pages": s["dev_pages"],
                     "trained on": s.get("categories", "all"), "held out": s.get("holdout_docs", "")})
if not runs:
    raise SystemExit("No finished runs yet. Run Notebook 2 first.")
runs = pd.DataFrame(runs).sort_values("mean of collections")
print(runs.round(3).to_string(index=False))

best = runs[runs["trained on"] == "all"].iloc[0]["run"] if (runs["trained on"] == "all").any() else runs.iloc[0]["run"]
USE = {                      # <-- the run for each collection's test pages
    "kade_letters": best,
    "survey_notes": best,
    "dominy_accounts": best,
}
for collection, run in USE.items():
    s = json.load(open(os.path.join(RUNS, run, "summary.json")))
    if s.get("categories", "all") != "all" and collection not in s["categories"].split(","):
        print(f"WARNING: {run} was trained on {s['categories']}, not {collection}.")
    if s.get("holdout_docs"):
        print(f"Note: {run} was trained with {s['holdout_docs']} held out. A *-final run is better for the test pages.")
print("\nUsing:", json.dumps(USE, indent=2))


## Step 4. Read the test pages

Each chosen run reads its collections' pages. The last number on each line is the time left.


In [ ]:
import hashlib

label = next(iter(USE.values())) if len(set(USE.values())) == 1 else \
    "combined-" + hashlib.md5(json.dumps(USE, sort_keys=True).encode()).hexdigest()[:8]
out_dir = os.path.join(WORK, "predictions", label)
os.makedirs(out_dir, exist_ok=True)
json.dump(USE, open(os.path.join(out_dir, "runs_used.json"), "w"), indent=2)
python = sys.executable
raws = []
for run in sorted(set(USE.values())):
    collections = ",".join(c for c, r in USE.items() if r == run)
    raw = os.path.join(out_dir, f"test_raw_{run}.csv")
    raws.append(raw)
    print(f"=== {run}: {collections}")
    !{python} {SCRIPTS}/transcribe.py --data_dir {DATA} --pages {DATA}/test_pages.csv --out {raw} --run_dir {os.path.join(RUNS, run)} --categories {collections}


## Step 5. Make `submission.csv`

The rows follow `sample_submission.csv` (or `test.csv` if there isn't one), with exactly the two
columns Kaggle accepts. It checks that every test page has text.

In [ ]:
preds = pd.concat([pd.read_csv(r, keep_default_na=False) for r in raws], ignore_index=True)
sample = os.path.join(DATA, "sample_submission.csv")
order = pd.read_csv(sample if os.path.exists(sample) else os.path.join(DATA, "test.csv"), keep_default_na=False)[["page_id"]]
submission = order.merge(preds[["page_id", "text"]], on="page_id", how="left").fillna({"text": ""})
empty = submission[submission.text.str.strip() == ""]
if len(empty):
    print(f"WARNING: {len(empty)} pages have no text, e.g. {', '.join(empty.page_id[:5])}. Run Step 4 again.")
path = os.path.join(out_dir, "submission.csv")
submission.to_csv(path, index=False)
print(f"{len(submission)} pages saved to {path}")
print(submission.assign(collection=submission.page_id.str.split("_").str[0]).collection.value_counts().to_string())
submission.head()

## Step 6. Download it, or save it to S3

- **Download:** in the file browser on the left, open `churro-work/predictions/<run>/`,
  right-click `submission.csv`, choose **Download**. Then upload it on the competition's
  **Submit Prediction** page.
- **Share with the team:** run the cell below.

Then **stop the instance** if you're done.

In [ ]:
MY_BUCKET = saved_setting("bucket")   # the bucket from Notebook 1; or type another, e.g. "my-bucket"

!aws s3 sync {out_dir} s3://{MY_BUCKET}/churro-work/predictions/{label} --quiet
print(f"Saved to s3://{MY_BUCKET}/churro-work/predictions/{label}")